# SiteGuard — train a PPE detector (YOLOv8) in Google Colab

**Goal:** fine-tune YOLOv8 to find workers without hard hats or safety vests, measure it honestly, and export `best.pt` for the SiteGuard API.

Before you start: **Runtime → Change runtime type → T4 GPU**.

Dataset: *Construction Site Safety Image Dataset* (originally from Roboflow Universe), mirrored on Kaggle as
`snehilsanyal/construction-site-safety-image-dataset-roboflow`. About 2,800 images, 10 classes, already in YOLO format.
Check the dataset licence on its Kaggle / Roboflow page and mention it in the README.

Run the cells top to bottom. Training takes roughly 1 hour on a free T4.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install libraries
!pip install -q ultralytics kagglehub

## 3. Download the dataset
`kagglehub` downloads public datasets. If it asks for credentials: kaggle.com → Settings → *Create New Token*,
then in Colab add secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` (key icon on the left) and re-run.

**Plan B:** download the zip yourself, upload it to Colab (folder icon on the left), run `!unzip -q your.zip -d /content/data`,
and set `DATA_ROOT = "/content/data"` instead of running this cell.

In [ ]:
import os
try:
    from google.colab import userdata
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except ImportError:
    pass

import kagglehub
DATA_ROOT = kagglehub.dataset_download("snehilsanyal/construction-site-safety-image-dataset-roboflow")
print("Downloaded to:", DATA_ROOT)

In [ ]:
# 4. Find the train/valid/test folders and write our own data.yaml
from pathlib import Path
import yaml

root = Path(DATA_ROOT)
def find_split(name):
    hits = [p for p in root.rglob("images") if p.parent.name.lower() == name]
    return hits[0] if hits else None

splits = {"train": find_split("train"), "val": find_split("valid") or find_split("val"), "test": find_split("test")}
print(splits)
assert splits["train"] and splits["val"], "Could not find train/valid folders - check DATA_ROOT"

DEFAULT_NAMES = ['Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest',
                 'Person', 'Safety Cone', 'Safety Vest', 'machinery', 'vehicle']
names = DEFAULT_NAMES
for y in root.rglob("*.yaml"):
    cfg = yaml.safe_load(y.read_text())
    if isinstance(cfg, dict) and "names" in cfg:
        names = list(cfg["names"].values()) if isinstance(cfg["names"], dict) else list(cfg["names"])
        print("Class names taken from", y)
        break
print(names)

data_cfg = {"train": str(splits["train"]), "val": str(splits["val"]), "names": {i: n for i, n in enumerate(names)}}
if splits["test"]:
    data_cfg["test"] = str(splits["test"])
Path("/content/siteguard_data.yaml").write_text(yaml.safe_dump(data_cfg, sort_keys=False))

## 5. Explore the data (EDA)
How many examples of each class are there? Rare classes (often the *NO-* ones) are harder to learn. This is **class imbalance**,
a good thing to discuss in your PS and at interview.

In [ ]:
import collections
import pandas as pd
import matplotlib.pyplot as plt

def count_labels(images_dir):
    labels_dir = images_dir.parent / "labels"
    counts, n_files = collections.Counter(), 0
    for f in labels_dir.glob("*.txt"):
        n_files += 1
        for line in f.read_text().splitlines():
            if line.strip():
                counts[names[int(line.split()[0])]] += 1
    return n_files, counts

rows = []
for split, d in splits.items():
    if d:
        n, c = count_labels(d)
        print(f"{split}: {n} images")
        rows += [{"split": split, "class": cls, "instances": c.get(cls, 0)} for cls in names]
eda = pd.DataFrame(rows)
train_counts = eda[eda.split == "train"].set_index("class")["instances"].sort_values()
train_counts.plot.barh(figsize=(7, 4), title="Training instances per class")
plt.tight_layout(); plt.savefig("/content/class_balance.png"); plt.show()
print("Most / least frequent class ratio:", round(train_counts.max() / max(train_counts.min(), 1), 1))

## 6. Train
Start from `yolov8n.pt` (smallest, fast on CPU later). If you have time, also try `yolov8s.pt` and compare — that comparison is itself a good result to report.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data="/content/siteguard_data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,      # stop early if validation stops improving
    seed=42,          # reproducible
    project="/content/runs",
    name="siteguard_v1",
    exist_ok=True,
)

## 7. Evaluate on the TEST split (images the model never saw)
These are the numbers you can honestly quote in your CV and PS.
- **Precision** — of the boxes the model reported, the share that were correct (few false alarms)
- **Recall** — of the real objects, the share the model found (few misses). Most important for safety
- **mAP@0.5** — overall detection quality, from 0 to 1

In [ ]:
import json
best = YOLO("/content/runs/siteguard_v1/weights/best.pt")
split = "test" if splits["test"] else "val"
m = best.val(data="/content/siteguard_data.yaml", split=split, plots=True)

per_class = []
for i, cls_idx in enumerate(m.box.ap_class_index):
    p, r, ap50, ap = m.box.class_result(i)
    per_class.append({"class": m.names[int(cls_idx)], "precision": round(float(p), 3), "recall": round(float(r), 3),
                      "mAP50": round(float(ap50), 3), "mAP50-95": round(float(ap), 3)})
per_class = pd.DataFrame(per_class)
display(per_class)

summary = {"split": split, "precision": round(float(m.box.mp), 3), "recall": round(float(m.box.mr), 3),
           "mAP50": round(float(m.box.map50), 3), "mAP50-95": round(float(m.box.map), 3),
           "per_class": per_class.to_dict(orient="records")}
print(json.dumps({k: v for k, v in summary.items() if k != "per_class"}, indent=2))

In [ ]:
# 8. Speed on CPU (what the Docker API will use)
import time, glob
test_imgs = sorted(glob.glob(str((splits["test"] or splits["val"]) / "*.jpg")))[:30]
best.predict(test_imgs[0], device="cpu", verbose=False)  # warm-up
t = time.perf_counter()
for p in test_imgs:
    best.predict(p, device="cpu", verbose=False)
cpu_ms = (time.perf_counter() - t) / len(test_imgs) * 1000
summary["cpu_ms_per_image"] = round(cpu_ms, 1)
print(f"CPU: {cpu_ms:.0f} ms per image (Colab CPU; your laptop may differ)")
Path("/content/metrics.json").write_text(json.dumps(summary, indent=2))

In [ ]:
# 9. Look at predictions with your own eyes
import random
from PIL import Image
for p in random.sample(test_imgs, min(4, len(test_imgs))):
    res = best.predict(p, conf=0.35, verbose=False)[0]
    display(Image.fromarray(res.plot()[:, :, ::-1]))

In [ ]:
# 10. Download the results. Put best.pt into the project's models/ folder,
#     and a few test images into samples/ for the camera simulator.
import shutil
out = Path("/content/siteguard_results"); out.mkdir(exist_ok=True)
run = Path("/content/runs/siteguard_v1")
shutil.copy(run / "weights/best.pt", out / "best.pt")
shutil.copy("/content/metrics.json", out / "metrics.json")
shutil.copy("/content/class_balance.png", out / "class_balance.png")
for f in ["results.png", "confusion_matrix.png", "PR_curve.png", "BoxPR_curve.png"]:
    if (run / f).exists():
        shutil.copy(run / f, out / f)
(out / "samples").mkdir(exist_ok=True)
for p in test_imgs[:20]:
    shutil.copy(p, out / "samples")
shutil.make_archive("/content/siteguard_results", "zip", out)
from google.colab import files
files.download("/content/siteguard_results.zip")

## 11. What to write afterwards (use YOUR numbers from metrics.json)
- **CV:** *Fine-tuned YOLOv8n on ~2,600 construction-site images; test mAP@0.5 = X, recall for NO-Hardhat = Y; ~Z ms per image on CPU.*
- **PS:** *The model reached a mAP@0.5 of X on unseen test images. It was weakest on [class], which is also the rarest class in the data.*

Put `results.png`, `confusion_matrix.png` and `class_balance.png` into the repo's `docs/` folder and show them in the README.